# 🤖 KNOT Intel Collector — Google Colab 버전

**사용법**
1. 상단 메뉴 → 런타임 → 모두 실행 (`Ctrl+F9`)
2. 마지막 셀에서 마크다운 보고서 내용 확인
3. 내용을 복사해서 Obsidian `01_inbox/intel-날짜.md` 에 붙여넣기

> **자동화를 원하면**: GitHub Actions 방식 사용 (`.github/workflows/intel_daily.yml`)

In [ ]:
# ─── 설정: 추적 소스 ──────────────────────────────────────────
import urllib.request, xml.etree.ElementTree as ET, datetime, textwrap

RSS_SOURCES = [
    {"name": "Karpathy - llm.c",      "url": "https://github.com/karpathy/llm.c/commits/master.atom",           "category": "karpathy"},
    {"name": "Karpathy - nanoGPT",    "url": "https://github.com/karpathy/nanoGPT/commits/master.atom",         "category": "karpathy"},
    {"name": "Anthropic Cookbook",    "url": "https://github.com/anthropics/anthropic-cookbook/commits/main.atom","category": "anthropic"},
    {"name": "Anthropic SDK Python",  "url": "https://github.com/anthropics/anthropic-sdk-python/releases.atom","category": "anthropic"},
    {"name": "Simon Willison (AI뉴스)","url": "https://simonwillison.net/atom/everything/",                      "category": "news"},
    {"name": "LangChain 릴리즈",       "url": "https://github.com/langchain-ai/langchain/releases.atom",         "category": "tools"},
]

def fetch(url):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "KNOT/1.0"})
        with urllib.request.urlopen(req, timeout=10) as r:
            return r.read().decode("utf-8", errors="ignore")
    except Exception as e:
        print(f"  ⚠️  {url[:50]} — {e}"); return None

def parse(xml, n=3):
    items = []
    try:
        root = ET.fromstring(xml)
        ns = "http://www.w3.org/2005/Atom"
        for e in root.findall(f"{{{ns}}}entry")[:n]:
            t = e.find(f"{{{ns}}}title");   title   = (t.text or "").strip() if t is not None else ""
            l = e.find(f"{{{ns}}}link");    link    = l.get("href","") if l is not None else ""
            u = e.find(f"{{{ns}}}updated"); updated = (u.text or "")[:10] if u is not None else ""
            s = e.find(f"{{{ns}}}summary") or e.find(f"{{{ns}}}content")
            summary = textwrap.shorten((s.text or "").strip(), 220) if s is not None else ""
            items.append({"title": title, "link": link, "date": updated, "summary": summary})
    except: pass
    return items

print("설정 완료 ✅")

In [ ]:
# ─── 수집 실행 ────────────────────────────────────────────────
today = datetime.date.today().isoformat()
all_items = []

for src in RSS_SOURCES:
    print(f"수집 중: {src['name']}")
    xml = fetch(src["url"])
    if not xml: continue
    items = parse(xml)
    for it in items:
        it["source_name"] = src["name"]
        it["category"]    = src["category"]
    all_items.extend(items)
    print(f"  → {len(items)}건")

print(f"\n총 {len(all_items)}건 수집 완료 ✅")

In [ ]:
# ─── 마크다운 보고서 생성 ─────────────────────────────────────
category_titles = {
    "karpathy":  "## 🧠 Andrej Karpathy",
    "anthropic": "## 🤖 Anthropic",
    "news":      "## 📰 AI 뉴스 큐레이터",
    "tools":     "## 🛠️ AI 도구 업데이트",
}

grouped = {}
for r in all_items:
    grouped.setdefault(r["category"], []).append(r)

lines = [
    f"---",
    f"date: {today}",
    f"type: intel-report",
    f"status: unread",
    f"---",
    f"",
    f"# AI 인텔 보고서 — {today}",
    f"> 수집 소스: Karpathy · Anthropic · AI 뉴스",
    f"",
]

for cat, heading in category_titles.items():
    if cat not in grouped: continue
    lines.append(heading); lines.append("")
    for it in grouped[cat]:
        lines.append(f"### [{it['title']}]({it['link']})")
        if it["date"]:    lines.append(f"- 날짜: `{it['date']}`")
        if it["summary"]: lines.append(f"- 요약: {it['summary']}")
        lines.append(f"- 소스: {it['source_name']}")
        lines.append("")

lines += [
    "---",
    "## 📝 블로그 소재 후보",
    "- [ ] ",
    "## 🔗 X 수동 확인",
    "- [ ] [@karpathy](https://x.com/karpathy)",
    "- [ ] [@AnthropicAI](https://x.com/AnthropicAI)",
]

report = "\n".join(lines)

print("="*60)
print(f"📄 01_inbox/intel-{today}.md  아래 내용을 복사하세요")
print("="*60)
print(report)

In [ ]:
# ─── (선택) Google Drive에 자동 저장 ─────────────────────────
# 아래 셀을 실행하면 Google Drive의 'KNOT/01_inbox/' 폴더에 자동 저장됩니다.

from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = "/content/drive/MyDrive/KNOT/01_inbox"
os.makedirs(save_dir, exist_ok=True)

save_path = f"{save_dir}/intel-{today}.md"
with open(save_path, "w", encoding="utf-8") as f:
    f.write(report)

print(f"✅ 저장 완료: {save_path}")